# MQTBench suite for Graphix simulators

To run this notebook, install the package with extra dependencies:

```bash
uv sync --extra examples
```

#### Benchmark characterization

As of version 0.3.5, Graphix supports two optimizations at the pattern level: _space minimization_ and _Pauli removal_.

- Space minimization rearranges the pattern commands in order to minimize the maximum number of qubits alive at any given time during the execution. This optimization is crucial to reduce the memory allocation in dense-state simulations. For patterns with causal flow (e.g., those directly transpiled from a quantum circuit), `Pattern.minimize_space` returns an optimal vale: `max_space = n_qubits + 1`.

- Pauli removal removes Pauli measurements on non-input qubits at the expense of adding local Clifford commands. This optimization can significantly reduce the number of commands (and, specially, measurement commands which are the bottleneck in dense-state simulations). However, it comes with a trade-off: patterns with causal flow are only guaranteed to have gflow after this optimization step. Performing space minimization on patterns without causal flow is known to be an NP-hard problem, and heuristics can fail to find a "good" measurement order.

As show in the table below, "Pauli removal + Space minimization" will often reduce the number of commands but `max_space` can become significantly larger than if only "Space minimization" is applied.

In [2]:
from graphix_mqtbench import generate_benchmarks
import pandas as pd

nqubit = 4
benchmarks = generate_benchmarks(nqubit)

rows = []
for bench in benchmarks:
    p = bench.pattern
    p_space = p.minimize_space(copy=True)
    p_pauli = p.infer_pauli_measurements().remove_pauli_measurements(copy=True)
    p_pauli_space = p_pauli.minimize_space(copy=True)

    rows.append({
        ("Circuit", "Benchmark"): bench.name.value,
        ("Circuit", "# Qubits"): bench.nqubits,
        ("Circuit", "# Gates"): len(bench.circuit.instruction),
        ("Transpilation", "Max Space"): p.max_space(),
        ("Transpilation", "# Commands"): len(p),
        ("Space min.", "Max Space"): p_space.max_space(),
        ("Space min.", "# Commands"): len(p_space),
        ("Pauli removal", "Max Space"): p_pauli.max_space(),
        ("Pauli removal", "# Commands"): len(p_pauli),
        ("Pauli removal + Space min.", "Max Space"): p_pauli_space.max_space(),
        ("Pauli removal + Space min.", "# Commands"): len(p_pauli_space),
    })

df = pd.DataFrame(rows)
df.columns = pd.MultiIndex.from_tuples(df.columns)
df

Circuit                  Transpilation             \
                  Benchmark # Qubits # Gates     Max Space # Commands   
0                        ae        4     128             8       1038   
1     bmw_quark_cardinality        4     123             6        876   
2          bmw_quark_copula        4      72             6        524   
3                        bv        4       4             6         17   
4   cdkm_ripple_carry_adder        4       7            22        182   
5                        dj        4      16             6        110   
6          draper_qft_adder        4      29             6        236   
7                full_adder        4       6            22        172   
8                       ghz        4       4             6         34   
9                graphstate        4       8             5         20   
10                   grover        4     132            22       1144   
11                      hhl        4      49             8        400   
12            modular_adder        4      31             6        236   
13               multiplier        4      42            22        430   
14                     qaoa        4      24             6        196   
15                      qft        4      34             6        280   
16             qftentangled        4      38             6        314   
17                      qnn        4      27             8        246   
18                 qpeexact        4      25             6        200   
19               qpeinexact        4      37             6        296   
20                    qwalk        4     250            22       2524   
21            randomcircuit        4     108            22       1052   
22        rg_qft_multiplier        4      40             6        336   
23   vbe_ripple_carry_adder        4       6            22        172   
24             vqe_real_amp        4      25             8        314   
25                  vqe_su2        4      89             6        730   
26            vqe_two_local        4      34             8        404   
27                   wstate        4      13             8        125   

   Space min.            Pauli removal            Pauli removal + Space min.  \
    Max Space # Commands     Max Space # Commands                  Max Space   
0           5        778            11        109                         10   
1           5        656            13        167                         10   
2           5        396            12        145                         12   
3           5         17             6         11                          5   
4           5        163            20         98                         20   
5           5         89             5         23                          5   
6           5        180            13         54                          9   
7           5        156            19         81                          8   
8           5         32             6         28                          5   
9           5         24             5         24                          5   
10          5        867            17        286                         11   
11          5        306            17         94                         16   
12          5        180            11         54                          9   
13          5        343            25        113                         11   
14          5        151             8         66                          8   
15          5        212            17         79                         11   
16          5        236            11         98                          9   
17          5        197             7         66                          5   
18          5        154             9         51                          9   
19          5        224            17         90                          8   
20          5       1919            84        643                    

#### Minimal backend benchmark

In [ ]:
import timeit
from graphix_mqtbench import MQTBenchmark, BenchmarkName
import numpy as np

rng = np.random.default_rng(42)

def simulate(pattern, backend):
    def run():
        return pattern.simulate_pattern(backend=backend, rng=rng)
    return run

benchmark = MQTBenchmark(name=BenchmarkName.QFT, nqubits=14)
pattern = benchmark.pattern.minimize_space()

run = simulate(pattern, backend="statevector")
timer = timeit.Timer(run)
t = min(timer.repeat(number=1, repeat=5))

print(
f"Benchmark = {benchmark.name.value}\n\
nqubits = {benchmark.nqubits}\n\
max_space = {pattern.max_space()}\n\
n_commands = {len(pattern)}\n\
simulation time = {t:.5f} s")

Benchmark = qft
nqubits = 14
max_space = 15
n_commands = 2982
simulation time = 1.57080 s
